# Week 3, Day 6: Forge: Week 3 challenge, quiz and deep read

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w003/d6.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np, pandas as pd
print("1.", pd.Series([1.0, np.nan, 3.0]).mean())
print("2.", np.sqrt(0.5 * 0.5 / 100))
print("3.", np.logspace(0, 2, 3))

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def psychometric(c, bias, sigma, lapse):
    return lapse + (1 - 2 * lapse) / (1 + np.exp(-(c - bias) / sigma))

rng = np.random.default_rng(2026)
contrasts = np.array([0, 0.0625, 0.125, 0.25, 1.0])
rows = []
for lab, lab_sigma in [("cshl", 0.14), ("ucl", 0.16), ("champalimaud", 0.15)]:
    for m in range(4):
        mouse = f"{lab}_{m + 1:02d}"
        bias = rng.normal(0, 0.08)
        sigma = abs(rng.normal(lab_sigma, 0.03)) + 0.05
        lapse = rng.uniform(0.02, 0.12)
        n = 500
        side = rng.choice([-1, 1], n); c = rng.choice(contrasts, n); signed = side * c
        chose_right = rng.random(n) < psychometric(signed, bias, sigma, lapse)
        rt = 0.25 + 0.6 * np.exp(-3 * c) * rng.lognormal(0, 0.35, n)
        rows.append(pd.DataFrame({"lab": lab, "mouse": mouse, "signed_contrast": signed, "chose_right": chose_right, "rt": rt}))
trials = pd.concat(rows, ignore_index=True)
print(trials.shape); trials.head()

### Exercise: Part A: fit every mouse

Write `fit_mouse(df)` that builds the psychometric table for one mouse (mean and count of `chose_right` per signed contrast), computes standard errors (at least 0.02), fits `psychometric` with `p0=[0, 0.2, 0.05]` and `bounds=([-1, 0.01, 0], [1, 1, 0.5])`, and returns a Series with `bias`, `sigma` and `lapse`. Apply it to every mouse to make `fits`, indexed by mouse, with a `lab` column.

In [ ]:
def fit_mouse(df):
    d = df.groupby("signed_contrast")["chose_right"].agg(["mean", "size"])
    se = ...
    p, _ = curve_fit(...)
    return pd.Series({"bias": p[0], "sigma": p[1], "lapse": p[2]})

fits = ...
fits

In [ ]:
import numpy as np
def test_fits_table():
    "fits has one row per mouse with lab, bias, sigma and lapse"
    assert len(fits) == 12, f"Expected 12 mice, found {len(fits)}."
    for col in ["lab", "bias", "sigma", "lapse"]:
        assert col in fits.columns, f"Missing column {col}."

def test_reasonable():
    "fitted values are in sensible ranges"
    assert fits["sigma"].between(0.01, 1).all() and fits["lapse"].between(0, 0.5).all()
    assert fits["bias"].abs().max() < 0.6

def test_matches_curve_fit():
    "the first mouse's bias matches a direct fit"
    m = fits.index[0]
    d = trials[trials["mouse"] == m].groupby("signed_contrast")["chose_right"].agg(["mean", "size"])
    se = np.maximum(np.sqrt(d["mean"] * (1 - d["mean"]) / d["size"]), 0.02)
    p, _ = curve_fit(psychometric, d.index.to_numpy(), d["mean"].to_numpy(), sigma=se.to_numpy(), p0=[0, 0.2, 0.05], bounds=([-1, 0.01, 0], [1, 1, 0.5]))
    assert abs(fits.loc[m, "bias"] - p[0]) < 1e-3

test_fits_table()
test_reasonable()
test_matches_curve_fit()
print("All checks passed")

### Exercise: Part B: the three-panel figure

Make `fig` with three panels side by side. A: for each lab, the pooled fraction choosing right at each contrast (points) and the curve with that lab's median parameters (line). B: each mouse's threshold `sigma`, as points grouped by lab. C: median reaction time against absolute contrast, one line per lab. Label every axis and add panel letters.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 3.4), layout="constrained")
cc = np.linspace(-1, 1, 200)
labs = fits["lab"].unique()

# A: psychometric data and median fit per lab

# B: thresholds per mouse, grouped by lab

# C: chronometric curve per lab

plt.show()

In [ ]:
def test_panels():
    "the figure has three panels"
    assert len(fig.axes) == 3, f"Expected 3 panels, found {len(fig.axes)}."

def test_labels():
    "every panel has labelled axes"
    for ax in fig.axes:
        assert ax.get_xlabel() and ax.get_ylabel(), "Label every axis."

def test_lines():
    "panel A has a fitted curve for each lab"
    assert len(fig.axes[0].get_lines()) >= 3, "Draw one fitted curve per lab in panel A."

test_panels()
test_labels()
test_lines()
print("All checks passed")